In [32]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [33]:
df = pd.read_csv('IPL.csv', low_memory=False)

In [34]:
df.head()

,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_runs,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,1,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,2,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


In [35]:
df.shape

(278205, 64)

In [36]:
df.dtypes



Unnamed: 0           int64
match_id             int64
date                object
match_type          object
event_name          object
                     ...  
batter_balls         int64
bowler_wicket        int64
batting_partners    object
next_batter         object
striker_out           bool
Length: 64, dtype: object

In [37]:
df.isnull().sum()

Unnamed: 0               0
match_id                 0
date                     0
match_type               0
event_name               0
                     ...  
batter_balls             0
bowler_wicket            0
batting_partners         0
next_batter         264884
striker_out              0
Length: 64, dtype: int64

In [38]:
df = df.drop(columns=['Unnamed: 0'])

In [39]:
important_cols = [
'match_id',
'date',
'season',
'batting_team',
'bowling_team',
'venue',
'over',
'ball',
'runs_total',
'team_runs',
'team_wicket',
'toss_winner',
'toss_decision',
'match_won_by'
]

df = df[important_cols]

In [40]:
df.columns

Index(['match_id', 'date', 'season', 'batting_team', 'bowling_team', 'venue',
       'over', 'ball', 'runs_total', 'team_runs', 'team_wicket', 'toss_winner',
       'toss_decision', 'match_won_by'],
      dtype='object')

In [41]:
df.head()
df.shape

(278205, 14)

In [42]:
df['date'] = pd.to_datetime(df['date'])

In [43]:
df.dtypes

match_id                  int64
date             datetime64[ns]
season                   object
batting_team             object
bowling_team             object
venue                    object
over                      int64
ball                      int64
runs_total                int64
team_runs                 int64
team_wicket               int64
toss_winner              object
toss_decision            object
match_won_by             object
dtype: object

In [44]:
df['balls_bowled'] = df['over']*6 + df['ball']
df['balls_remaining'] = 120 - df['balls_bowled']

In [45]:
df[['over','ball','balls_bowled','balls_remaining']].head()

,over,ball,balls_bowled,balls_remaining
0,0,1,1,119
1,0,2,2,118
2,0,3,3,117
3,0,3,3,117
4,0,4,4,116


In [46]:
df['wickets_left'] = 10 - df['team_wicket']

In [ ]:
df['current_run_rate'] = df['team_runs'] / (df['balls_bowled']/6)
df['current_run_rate'] = df['current_run_rate'].replace([np.inf, -np.inf], 0)

In [48]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

In [29]:
df.head()

,match_id,date,season,batting_team,bowling_team,venue,over,ball,runs_total,team_runs,team_wicket,toss_winner,toss_decision,match_won_by,balls_bowled,balls_remaining,wickets_left,current_run_rate
0,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,0,1,1,1,0,Royal Challengers Bangalore,field,Kolkata Knight Riders,1,119,10,6.0
1,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,0,2,0,1,0,Royal Challengers Bangalore,field,Kolkata Knight Riders,2,118,10,3.0
2,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,0,3,1,2,0,Royal Challengers Bangalore,field,Kolkata Knight Riders,3,117,10,4.0
3,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,0,3,0,2,0,Royal Challengers Bangalore,field,Kolkata Knight Riders,3,117,10,4.0
4,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,0,4,0,2,0,Royal Challengers Bangalore,field,Kolkata Knight Riders,4,116,10,3.0


In [49]:
first_innings_score = df.groupby('match_id')['team_runs'].max().reset_index()
first_innings_score.rename(columns={'team_runs':'target'}, inplace=True)
first_innings_score['target'] = first_innings_score['target'] + 1

In [50]:
df = df.merge(first_innings_score, on='match_id')

In [51]:
df['runs_remaining'] = df['target'] - df['team_runs']

In [52]:
df['overs_remaining'] = df['balls_remaining'] / 6

In [ ]:
df['required_run_rate'] = df['runs_remaining'] / df['overs_remaining']
df['required_run_rate'] = df['required_run_rate'].replace([np.inf, -np.inf], 0)

In [55]:
df.to_csv("cleaned_ipl.csv", index=False)